# R5 — Claims Ops dashboard (dataset prep)

The dashboard itself is authored on the AI/BI dashboard page; this notebook holds the **dataset queries** it is built from. Every dataset reads **only the trusted layer** (`claims.analytics` star schema) — nothing outside it.

Dashboard spec (adjusters + managers, no SQL):
* **4 counters** — total claims, approved amount, rejection rate (rejected ÷ decided), average turnaround days.
* **Monthly trend** — claim volume + billed amount by month.
* **Hospital bar chart** — total paid by hospital (settled claims).
* **Rejection-code breakdown** — rejected claims per rejection code.
* **Filters** — `plan_code`, `city_tier`, and a date range, all responding together.

In [0]:
%sql
-- Counter values (single row): total claims, approved amount, rejection rate over DECIDED, avg turnaround
SELECT
  COUNT(*)                                                     AS total_claims,
  ROUND(SUM(approved_amount), 2)                               AS approved_amount,
  ROUND(SUM(CASE WHEN status = 'rejected' THEN 1 ELSE 0 END)
        / SUM(CASE WHEN status IN ('approved','rejected','settled') THEN 1 ELSE 0 END), 4) AS rejection_rate,
  ROUND(AVG(turnaround_days), 2)                               AS avg_turnaround_days
FROM claims.analytics.fact_claims;

In [0]:
%sql
-- Monthly trend: claim volume + billed amount (admission month)
SELECT
  d.month_start,
  COUNT(*)                       AS claim_count,
  ROUND(SUM(f.billed_amount), 2) AS total_billed
FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_date d ON f.admission_date_key = d.date_key
GROUP BY d.month_start
ORDER BY d.month_start;

In [0]:
%sql
-- Hospital spend bar chart: total paid (settled only)
SELECT
  h.hospital_id, h.hospital_name, h.city_tier,
  ROUND(SUM(f.approved_amount), 2) AS total_paid
FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_hospital h ON f.hospital_sk = h.hospital_sk
WHERE f.status = 'settled'
GROUP BY h.hospital_id, h.hospital_name, h.city_tier
ORDER BY total_paid DESC;

In [0]:
%sql
-- Rejection-code breakdown
SELECT
  f.rejection_code,
  COUNT(*) AS rejected_count
FROM claims.analytics.fact_claims f
WHERE f.status = 'rejected'
GROUP BY f.rejection_code
ORDER BY rejected_count DESC;

In [0]:
%sql
-- Filter lookup datasets: distinct plan codes and city tiers for the dashboard filters
SELECT DISTINCT p.plan_code FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_plan p ON f.plan_sk = p.plan_sk ORDER BY p.plan_code;

SELECT DISTINCT h.city_tier FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_hospital h ON f.hospital_sk = h.hospital_sk ORDER BY h.city_tier;